# ARC repair SFT

This notebook trains a completion-only LoRA on genuine repair failures, 15% ordinary solve replay, and 1% zero-mask no-ops. The previous wrong assistant candidate is context only and receives no loss. One `<REPAIR>` token is added and initialized from the mean of the existing structural-token embeddings.


In [ ]:
RUN_EXPERIMENT = True
DATASET_SLUG = 'arc26-repair-failures-18432'
MAX_TRAIN_EXAMPLES = None
EPOCHS = 1.0
DIAGNOSTIC_EXAMPLES = 64
ROLLOUT_EXAMPLES = 16
EXPECTED_SOURCE_HASHES = {'repair_mining.py': 'a760956f0cff78fc4599250ccc0b90aa46aefc4feaa0094f2a9b6eb0dd2b13e6', 'repair_sft.py': 'c044b97afbca2f69199901103bc98d96287793485a5384eb4056071bdf532fcc', 'train_repair_adapter.py': 'c3767e55753db7978b35d647ace638099aa67ef93c63975ef56deb2c0f72c1bd', 'arc_solver.py': '18016acbf71f9ba6e62c32f5a3fcb581488f0934cb58e7f36e177286992e4896'}

MODEL_PATH = '/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1'
OUTPUT_DIR = '/kaggle/working/repair_sft_full'

print('dataset =', DATASET_SLUG, 'max train =', MAX_TRAIN_EXAMPLES, 'epochs =', EPOCHS)


In [ ]:
if RUN_EXPERIMENT:
    import hashlib
    import subprocess
    from pathlib import Path

    gpu_names = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        text=True,
    ).strip().splitlines()
    if len(gpu_names) != 4 or any('L4' not in name for name in gpu_names):
        raise RuntimeError(
            f'Repair SFT requires the pinned 4xL4 environment; observed GPUs={gpu_names}'
        )
    print('verified GPUs =', gpu_names)

    code_candidates = [
        Path('/kaggle/input/datasets/yuvraj/arc2026/ARC-AGI1/qwen_baseline'),
        Path('/kaggle/input/arc2026/ARC-AGI1/qwen_baseline'),
    ]
    CODE_DIR = next((path for path in code_candidates if path.is_dir()), None)
    if CODE_DIR is None:
        raise FileNotFoundError(f'Could not find arc2026 code under {code_candidates}')
    observed_hashes = {
        name: hashlib.sha256((CODE_DIR / name).read_bytes()).hexdigest()
        for name in EXPECTED_SOURCE_HASHES
    }
    if observed_hashes != EXPECTED_SOURCE_HASHES:
        raise RuntimeError(
            'Mounted arc2026 code is stale or unexpected. '
            f'expected={EXPECTED_SOURCE_HASHES} observed={observed_hashes}'
        )

    data_candidates = [
        Path('/kaggle/input/datasets/yuvraj') / DATASET_SLUG,
        Path('/kaggle/input') / DATASET_SLUG,
    ]
    DATA_DIR = next((path for path in data_candidates if path.is_dir()), None)
    if DATA_DIR is None:
        raise FileNotFoundError(f'Could not find repair dataset under {data_candidates}')
    print('verified code =', CODE_DIR)
    print('repair data =', DATA_DIR)


In [ ]:
if RUN_EXPERIMENT:
    import os
    import subprocess
    import sys

    os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
    os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
    os.environ['OMP_NUM_THREADS'] = '3'
    if not Path(os.environ['TRITON_PTXAS_PATH']).is_file():
        raise FileNotFoundError(os.environ['TRITON_PTXAS_PATH'])

    command = [
        sys.executable,
        '-m', 'torch.distributed.run',
        '--standalone',
        '--nproc_per_node', '4',
        str(CODE_DIR / 'train_repair_adapter.py'),
        '--model-path', MODEL_PATH,
        '--train-path', str(DATA_DIR / 'repair_failures.train.jsonl'),
        '--dev-path', str(DATA_DIR / 'repair_failures.dev.jsonl'),
        '--output-dir', OUTPUT_DIR,
        '--epochs', str(EPOCHS),
        '--lora-rank', '256',
        '--gradient-accumulation-steps', '1',
        '--expected-world-size', '4',
        '--solve-replay-fraction', '0.15',
        '--noop-fraction', '0.01',
        '--diagnostic-examples', str(DIAGNOSTIC_EXAMPLES),
        '--rollout-examples', str(ROLLOUT_EXAMPLES),
    ]
    if MAX_TRAIN_EXAMPLES is not None:
        command.extend(['--max-train-examples', str(MAX_TRAIN_EXAMPLES)])
    print('running:', ' '.join(command))
    subprocess.run(command, cwd=str(CODE_DIR), check=True)


In [ ]:
if RUN_EXPERIMENT:
    import json
    from pathlib import Path

    manifest_path = Path(OUTPUT_DIR) / 'repair_sft_manifest.json'
    manifest = json.loads(manifest_path.read_text())
    assert manifest['tokenizer']['old_vocab_size'] == 16
    assert manifest['tokenizer']['new_vocab_size'] == 17
    assert manifest['tokenizer']['repair_token_id'] == 16
    assert manifest['config']['lora_rank'] == 256
    assert manifest['distributed']['world_size'] == 4
    assert manifest['distributed']['effective_global_batch_size'] == 4
    assert manifest['mixture']['requested_fractions'] == {
        'repair_failure': 0.84,
        'solve_replay': 0.15,
        'repair_noop': 0.01,
    }
    assert manifest['adapter_update']['after']['nonzero_elements'] > 0
    print(json.dumps(manifest, indent=2))
